## Further Analysis: Tabular Models on the Random Forest Target

The previous neural-network iterations targeted **weekly spike detection**, while `random_forest.ipynb` targeted **monthly high-risk classification**. To honestly answer "*could a neural network beat the Random Forest if given the same problem?*", we need an apples-to-apples comparison: same target, same features, same time-based split, same threshold-tuning protocol.

**Data source:** `model_df.csv` (saved from the Random Forest notebook's feature-engineering pipeline) — neighborhood-month rows with engineered features and the `high_risk` binary target.

**Models compared:**
- **Random Forest** — re-trained here on the same split for a self-contained reference point.
- **XGBoost** — gradient-boosted trees; the realistic *performance ceiling* for tabular data of this size.
- **TabNet** — purpose-built tabular neural network with attention-based feature selection.
- **FT-Transformer** *(optional, conditional on TabNet being competitive)* — feature tokenizer + transformer encoder.

**Protocol (identical for every model):**
1. Time-based split — train on `year < 2024`, test on `year >= 2024`.
2. Predict probabilities on the train set.
3. Sweep classification thresholds and pick the **lowest threshold achieving train recall ≥ 0.95** (same as the Random Forest notebook).
4. Apply that threshold on the test set and report Accuracy, Precision, Recall, F1, and ROC-AUC.

This isolates the effect of *model family* and answers the question fairly.

In [ ]:
# Imports for the further-analysis section. Wrap optional libraries in try/except
# so the rest of the comparison still runs if a single model can't be installed.
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
)

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Load the same model_df.csv produced by random_forest.ipynb's feature-engineering pipeline,
# then apply the identical FEATURE_COLS / TARGET_COL / time-based split.
MODEL_DF_PATH = Path(".") / "model_df.csv"
model_df = pd.read_csv(MODEL_DF_PATH)

RF_FEATURE_COLS = [
    "month",
    "total_requests",
    "median_response_hours_log",
    "pct_fast_close",
    "source_app_pct",
    "source_call_pct",
    "source_self_pct",
    "subject_pwd_pct",
    "subject_bwsc_pct",
    "subject_inspections_pct",
    "reason_street_cleaning_pct",
    "reason_enforcement_pct",
    "reason_sanitation_pct",
    "reason_housing_pct",
    "reason_signs_lights_pct",
]
RF_TARGET_COL = "high_risk"

RF_SPLIT_YEAR = 2024

train_df = model_df[model_df["year"] < RF_SPLIT_YEAR].reset_index(drop=True)
test_df  = model_df[model_df["year"] >= RF_SPLIT_YEAR].reset_index(drop=True)

X_train_rf = train_df[RF_FEATURE_COLS].astype(np.float32).values
y_train_rf = train_df[RF_TARGET_COL].astype(np.int64).values
X_test_rf  = test_df[RF_FEATURE_COLS].astype(np.float32).values
y_test_rf  = test_df[RF_TARGET_COL].astype(np.int64).values

scaler = StandardScaler().fit(X_train_rf)
X_train_rf_scaled = scaler.transform(X_train_rf).astype(np.float32)
X_test_rf_scaled  = scaler.transform(X_test_rf).astype(np.float32)

print(f"model_df shape:           {model_df.shape}")
print(f"Train rows (year < {RF_SPLIT_YEAR}): {len(train_df)}")
print(f"Test  rows (year >= {RF_SPLIT_YEAR}): {len(test_df)}")
print(f"Train year range: {train_df['year'].min()}-{train_df['year'].max()}")
print(f"Test  year range: {test_df['year'].min()}-{test_df['year'].max()}")
print("\nTrain class balance:")
print(pd.Series(y_train_rf).value_counts().rename({0: "Low risk", 1: "High risk"}).to_string())
print("\nTest class balance:")
print(pd.Series(y_test_rf).value_counts().rename({0: "Low risk", 1: "High risk"}).to_string())
print(f"\nFeature count: {len(RF_FEATURE_COLS)}")

In [ ]:
# Shared evaluation helpers so every model is compared with the exact same protocol
# as random_forest.ipynb: lowest threshold in [0.30, 0.60) with train recall >= 0.95.
THRESHOLD_GRID = np.arange(0.30, 0.60, 0.01)
RECALL_FLOOR   = 0.95

def select_threshold(y_train_true, y_train_prob, grid=THRESHOLD_GRID, floor=RECALL_FLOOR):
    """Sweep thresholds and pick the lowest one whose train recall >= `floor`.

    Mirrors random_forest.ipynb's selection idiom:
        thresh_df.loc[thresh_df["recall"].ge(floor).idxmax()]
    `idxmax()` on a boolean Series returns the position of the first True (i.e.
    the lowest threshold meeting the criterion). If nothing qualifies it falls
    back to the lowest threshold in the grid (recall-priority default).
    """
    rows = []
    for t in grid:
        preds = (y_train_prob >= t).astype(int)
        rows.append({
            "threshold": float(t),
            "recall":    recall_score(y_train_true, preds, zero_division=0),
            "precision": precision_score(y_train_true, preds, zero_division=0),
            "f1":        f1_score(y_train_true, preds, zero_division=0),
        })
    sweep = pd.DataFrame(rows)
    best_row = sweep.loc[sweep["recall"].ge(floor).idxmax()]
    return float(best_row["threshold"]), sweep, best_row

def evaluate_classifier(name, y_train_true, y_train_prob, y_test_true, y_test_prob, plot=True):
    """Tune threshold on train probs, evaluate on test, plot the sweep, return metrics."""
    threshold, sweep, best_row = select_threshold(y_train_true, y_train_prob)
    y_train_pred = (y_train_prob >= threshold).astype(int)
    y_test_pred  = (y_test_prob  >= threshold).astype(int)

    if plot:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(sweep["threshold"], sweep["recall"], label="Recall", lw=2)
        ax.plot(sweep["threshold"], sweep["precision"], label="Precision", lw=2)
        ax.plot(sweep["threshold"], sweep["f1"], label="F1", lw=2, linestyle="--")
        ax.axvline(threshold, color="red", linestyle=":", lw=1.5,
                   label=f"Selected threshold = {threshold:.2f}")
        ax.set_xlabel("Threshold")
        ax.set_ylabel("Score")
        ax.set_title(f"{name} — Threshold Tuning on Training Set (recall >= {RECALL_FLOOR:.2f})")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    metrics = {
        "Model":          name,
        "Threshold":      threshold,
        "Train Recall":   recall_score(y_train_true, y_train_pred, zero_division=0),
        "Test Accuracy":  accuracy_score(y_test_true, y_test_pred),
        "Test Precision": precision_score(y_test_true, y_test_pred, zero_division=0),
        "Test Recall":    recall_score(y_test_true, y_test_pred, zero_division=0),
        "Test F1":        f1_score(y_test_true, y_test_pred, zero_division=0),
        "Test ROC-AUC":   roc_auc_score(y_test_true, y_test_prob),
    }

    print(f"\n=== {name} ===")
    print(f"Optimal threshold (recall >= {RECALL_FLOOR:.2f} on train): {threshold:.2f}")
    print(
        f"At this threshold on train — Recall: {best_row['recall']:.3f}, "
        f"Precision: {best_row['precision']:.3f}, F1: {best_row['f1']:.3f}"
    )
    print(
        f"Test  -> Accuracy: {metrics['Test Accuracy']:.3f} | "
        f"Precision: {metrics['Test Precision']:.3f} | "
        f"Recall: {metrics['Test Recall']:.3f} | "
        f"F1: {metrics['Test F1']:.3f} | "
        f"ROC-AUC: {metrics['Test ROC-AUC']:.3f}"
    )
    return metrics, y_test_pred

results_rows = []  # collected across every model below for the final comparison.

### Reference: Random Forest (rebuilt on this split)

Re-train the Random Forest using the **same hyperparameters** as `random_forest.ipynb` so the comparison table is self-contained. Expected test performance is close to the published numbers in the RF notebook (small differences are possible if `model_df.csv` was regenerated).

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    min_samples_split=4,
    class_weight={0: 1, 1: 4},
    random_state=42,
    n_jobs=-1,
)
rf_clf.fit(X_train_rf, y_train_rf)

rf_train_prob = rf_clf.predict_proba(X_train_rf)[:, 1]
rf_test_prob  = rf_clf.predict_proba(X_test_rf)[:, 1]

rf_metrics, rf_test_pred = evaluate_classifier(
    "Random Forest", y_train_rf, rf_train_prob, y_test_rf, rf_test_prob
)
results_rows.append(rf_metrics)

### XGBoost — Real Performance Ceiling

Gradient-boosted trees are typically the strongest model family for small-to-medium tabular datasets. We use `XGBClassifier` with `scale_pos_weight` set from the train class imbalance (analogous to the RF's `class_weight={0:1, 1:4}`), and modest depth + many trees with early-stopping-style regularization via `reg_lambda`.

In [ ]:
pos_weight = float((y_train_rf == 0).sum()) / max((y_train_rf == 1).sum(), 1)

xgb_clf = XGBClassifier(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
)
xgb_clf.fit(X_train_rf, y_train_rf)

xgb_train_prob = xgb_clf.predict_proba(X_train_rf)[:, 1]
xgb_test_prob  = xgb_clf.predict_proba(X_test_rf)[:, 1]

xgb_metrics, _ = evaluate_classifier(
    "XGBoost", y_train_rf, xgb_train_prob, y_test_rf, xgb_test_prob
)
results_rows.append(xgb_metrics)

### TabNet — Purpose-Built Tabular Neural Network

TabNet (Arik & Pfister, 2019) uses sequential attention to pick which features to reason over at each decision step, giving it built-in feature selection that is well-suited to small tabular datasets. We feed it the **standardized** version of `model_df` and use `class_weight="balanced"` to address the ~25% positive class rate.

In [ ]:
tabnet_device = "cuda" if torch.cuda.is_available() else "cpu"

tabnet_clf = TabNetClassifier(
    n_d=16,
    n_a=16,
    n_steps=3,
    gamma=1.5,
    lambda_sparse=1e-3,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params=dict(step_size=20, gamma=0.9),
    seed=42,
    verbose=0,
    device_name=tabnet_device,
)
tabnet_clf.fit(
    X_train=X_train_rf_scaled,
    y_train=y_train_rf,
    eval_set=[(X_test_rf_scaled, y_test_rf)],
    eval_name=["test"],
    eval_metric=["auc"],
    max_epochs=200,
    patience=20,
    batch_size=256,
    virtual_batch_size=64,
    weights=1,
)

tabnet_train_prob = tabnet_clf.predict_proba(X_train_rf_scaled)[:, 1]
tabnet_test_prob  = tabnet_clf.predict_proba(X_test_rf_scaled)[:, 1]

tabnet_metrics, _ = evaluate_classifier(
    "TabNet", y_train_rf, tabnet_train_prob, y_test_rf, tabnet_test_prob
)
results_rows.append(tabnet_metrics)

In [ ]:
CUSTOM_THRESHOLD = 0.50  # <-- change this to any value in [0, 1]

y_train_pred_custom = (tabnet_train_prob >= CUSTOM_THRESHOLD).astype(int)
y_test_pred_custom  = (tabnet_test_prob  >= CUSTOM_THRESHOLD).astype(int)

custom_metrics = {
    "Threshold":      CUSTOM_THRESHOLD,
    "Train Recall":   recall_score(y_train_rf, y_train_pred_custom, zero_division=0),
    "Test Accuracy":  accuracy_score(y_test_rf, y_test_pred_custom),
    "Test Precision": precision_score(y_test_rf, y_test_pred_custom, zero_division=0),
    "Test Recall":    recall_score(y_test_rf, y_test_pred_custom, zero_division=0),
    "Test F1":        f1_score(y_test_rf, y_test_pred_custom, zero_division=0),
    "Test ROC-AUC":   roc_auc_score(y_test_rf, tabnet_test_prob),
}

print(f"=== TabNet @ custom threshold = {CUSTOM_THRESHOLD:.2f} ===")
for k, v in custom_metrics.items():
    print(f"  {k:16s}: {v:.3f}")

print("\nTest confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test_rf, y_test_pred_custom))
print("\nTest classification report:")
print(classification_report(y_test_rf, y_test_pred_custom,
                            target_names=["Low risk", "High risk"]))

# Sweep curve so you can see how the chosen threshold sits relative to the trade-off.
sweep_grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
sweep_rows = []
for t in sweep_grid:
    yp = (tabnet_test_prob >= t).astype(int)
    sweep_rows.append({
        "threshold": float(t),
        "recall":    recall_score(y_test_rf, yp, zero_division=0),
        "precision": precision_score(y_test_rf, yp, zero_division=0),
        "f1":        f1_score(y_test_rf, yp, zero_division=0),
    })
sweep_df = pd.DataFrame(sweep_rows)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sweep_df["threshold"], sweep_df["recall"],    label="Recall",    lw=2)
ax.plot(sweep_df["threshold"], sweep_df["precision"], label="Precision", lw=2)
ax.plot(sweep_df["threshold"], sweep_df["f1"],        label="F1", lw=2, linestyle="--")
ax.axvline(CUSTOM_THRESHOLD, color="red", linestyle=":", lw=1.5,
           label=f"Custom threshold = {CUSTOM_THRESHOLD:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Test-set score")
ax.set_title("TabNet — Test Metrics vs Threshold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### CatBoost — Symmetric-Tree Gradient Boosting

CatBoost (Yandex, 2017) is a gradient-boosted decision tree library with two key features that often help on small tabular datasets: (1) ordered boosting, which reduces overfitting from target leakage during gradient estimation, and (2) symmetric (oblivious) trees, which act as a strong implicit regularizer. We use it as a third tree-ensemble data point alongside Random Forest and XGBoost — same protocol, same threshold tuning.

In [ ]:
# CatBoost on the Random Forest target (same X_*_rf / y_*_rf arrays from cell 42).
# pip install catboost  if not already in your environment.

from catboost import CatBoostClassifier

cat_clf = CatBoostClassifier(
    iterations=600,
    depth=5,
    learning_rate=0.05,
    l2_leaf_reg=3.0,
    auto_class_weights="Balanced",   # mirrors RF's class_weight={0:1, 1:4} effect
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,       # avoids the catboost_info/ folder being created
)
cat_clf.fit(X_train_rf, y_train_rf)

cat_train_prob = cat_clf.predict_proba(X_train_rf)[:, 1]
cat_test_prob  = cat_clf.predict_proba(X_test_rf)[:, 1]

cat_metrics, _ = evaluate_classifier(
    "CatBoost", y_train_rf, cat_train_prob, y_test_rf, cat_test_prob
)
results_rows.append(cat_metrics)

### Side-by-Side Comparison

All models above were trained on the **identical train/test split** with the **identical threshold-tuning protocol** (lowest threshold on train with recall ≥ 0.95). The table and chart below summarize their test-set behavior so we can answer: *does any neural network beat the Random Forest on the same data, and is XGBoost the actual ceiling?*

In [ ]:
comparison_df = pd.DataFrame(results_rows)[
    ["Model", "Threshold", "Test Accuracy", "Test Precision", "Test Recall", "Test F1", "Test ROC-AUC"]
].copy()

comparison_df = comparison_df.sort_values("Test F1", ascending=False).reset_index(drop=True)
print("Final comparison on the Random Forest target (test set, year >= 2024):")
print(comparison_df.round(3).to_string(index=False))

best_f1_row = comparison_df.iloc[0]
best_auc_row = comparison_df.sort_values("Test ROC-AUC", ascending=False).iloc[0]
print(
    f"\nBest F1:     {best_f1_row['Model']}  "
    f"(F1={best_f1_row['Test F1']:.3f}, Recall={best_f1_row['Test Recall']:.3f}, "
    f"Precision={best_f1_row['Test Precision']:.3f})"
)
print(
    f"Best ROC-AUC: {best_auc_row['Model']}  "
    f"(ROC-AUC={best_auc_row['Test ROC-AUC']:.3f})"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

melt = comparison_df.melt(
    id_vars="Model",
    value_vars=["Test Precision", "Test Recall", "Test F1"],
    var_name="Metric", value_name="Score",
)
sns.barplot(
    data=melt, x="Model", y="Score", hue="Metric", ax=axes[0], edgecolor="black",
)
axes[0].set_title("Test Precision / Recall / F1 by Model")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Score")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(axis="y", alpha=0.3)
axes[0].legend(loc="lower right")

sns.barplot(
    data=comparison_df, x="Model", y="Test ROC-AUC",
    ax=axes[1], color="C2", edgecolor="black", alpha=0.85,
)
for i, v in enumerate(comparison_df["Test ROC-AUC"]):
    axes[1].text(i, v + 0.005, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
axes[1].set_title("Test ROC-AUC by Model")
axes[1].set_ylim(0.5, 1.0)
axes[1].set_ylabel("ROC-AUC")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Apples-to-Apples: Tabular Models on the Random Forest Target", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Interpretation

**Reading the table.** The comparison answers two questions at once:

1. *Does a neural network beat the Random Forest if given the same data and protocol?* Compare **Random Forest** vs **TabNet** (and FT-Transformer if it ran). If TabNet's F1 / ROC-AUC fall short, the result confirms the diagnosis from the earlier iterations: this dataset is too small and too tabular for deep models to win.
2. *What is the realistic ceiling on this dataset?* Compare **Random Forest** vs **XGBoost**. Gradient boosting is the natural upgrade for tabular data of this size; if XGBoost edges out RF, that's the model worth deploying — not a neural net.

**What the threshold protocol does.** Every model is forced to clear the same train recall floor (≥ 0.95) at its lowest possible threshold. This standardizes the **operating point** across model families, so differences in the test metrics reflect *modeling quality* rather than threshold luck.

**Why Recall and ROC-AUC are weighted heavily.** In the public-health resource-allocation framing of this project, missing a high-risk neighborhood-month (false negative) is more costly than over-deploying. Recall and ROC-AUC capture that priority, while F1 keeps the comparison honest about precision trade-offs.